In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
from src.data.repositories.sqlite_stock_repository import SqliteStockRepository


DEFAULT_DB_PATH = Path("src/stocks.db")

repository = SqliteStockRepository(str(DEFAULT_DB_PATH))

In [ ]:
import numpy as np
import pandas as pd


def ma_crossover_signals(
    close: pd.Series, short_window: int, long_window: int
) -> pd.DataFrame:
    close = close.astype(float)

    ma_s = close.rolling(short_window, min_periods=short_window).mean()
    ma_l = close.rolling(long_window, min_periods=long_window).mean()

    bull = (ma_s > ma_l) & (ma_s.shift(1) <= ma_l.shift(1))
    bear = (ma_s < ma_l) & (ma_s.shift(1) >= ma_l.shift(1))

    out = pd.DataFrame(
        {
            "close": close,
            "ma_short": ma_s,
            "ma_long": ma_l,
            "bull_cross": bull.fillna(False),
            "bear_cross": bear.fillna(False),
        },
        index=close.index,
    )
    return out


def backtest_ma_crossover(
    df: pd.DataFrame,
    short_window: int = 10,
    long_window: int = 30,
    long_only: bool = True,
    fee_bps: float = 10.0,
    initial_cash: float = 10_000.0,
) -> dict:
    close = df["Close"]
    s = ma_crossover_signals(close, short_window, long_window)

    desired = pd.Series(0.0, index=s.index)

    if long_only:
        desired[s["bull_cross"]] = 1.0
        desired[s["bear_cross"]] = 0.0
        desired = desired.replace(0.0, np.nan).ffill().fillna(0.0)
    else:
        desired[s["bull_cross"]] = 1.0
        desired[s["bear_cross"]] = -1.0
        desired = desired.replace(0.0, np.nan).ffill().fillna(0.0)

    position = desired.shift(1).fillna(0.0)

    ret = close.pct_change().fillna(0.0)
    strat_gross = position * ret

    fee = fee_bps / 10_000.0
    turnover = position.diff().abs().fillna(0.0)
    strat_net = strat_gross - turnover * fee

    equity = (1.0 + strat_net).cumprod() * initial_cash

    trades = pd.DataFrame(
        {
            "position": position,
            "turnover": turnover,
            "ret": ret,
            "strat_net": strat_net,
            "equity": equity,
        },
        index=df.index,
    )

    metrics = compute_metrics(trades["strat_net"], trades["equity"])

    return {
        "signals": s,
        "position": position,
        "trades": trades,
        "metrics": metrics,
    }


def compute_metrics(
    strategy_returns: pd.Series, equity: pd.Series, periods_per_year: int = 252
) -> dict:
    r = strategy_returns.astype(float)
    eq = equity.astype(float)

    total_return = (eq.iloc[-1] / eq.iloc[0]) - 1.0

    ann_return = (1.0 + r.mean()) ** periods_per_year - 1.0
    ann_vol = r.std(ddof=0) * np.sqrt(periods_per_year)

    sharpe = np.nan
    if ann_vol and ann_vol > 0:
        sharpe = ann_return / ann_vol

    peak = eq.cummax()
    dd = eq / peak - 1.0
    max_dd = dd.min()

    win_rate = (r[r != 0] > 0).mean() if (r != 0).any() else np.nan

    return {
        "total_return": float(total_return),
        "annualized_return": float(ann_return),
        "annualized_vol": float(ann_vol),
        "sharpe": float(sharpe) if sharpe == sharpe else np.nan,
        "max_drawdown": float(max_dd),
        "win_rate_on_active_days": float(win_rate) if win_rate == win_rate else np.nan,
    }

In [10]:
df = repository.load_history("AAPL", "1d")[:-200]

/Users/tomekogiolda/Projects/agh/sem-2/evolutionary-algorithms/project/src/data/repositories/sqlite_stock_repository.py:82: FutureWarning:

In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`



In [26]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_stocks_and_revenue(
    df, short_window=10, long_window=30, title="MA Crossover + Returns"
):
    res = backtest_ma_crossover(
        df=df,
        short_window=short_window,
        long_window=long_window,
        long_only=True,
        fee_bps=10.0,
        initial_cash=10_000.0,
    )

    signals = res["signals"]
    trades = res["trades"]

    equity = trades["equity"]
    cum_return = equity / equity.iloc[0] - 1.0

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        row_heights=[0.8, 0.2],
    )

    fig.add_trace(
        go.Candlestick(
            x=df.index,
            open=df["Open"],
            high=df["High"],
            low=df["Low"],
            close=df["Close"],
            name="AAPL",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=signals.index,
            y=signals["ma_short"],
            mode="lines",
            name="MA short",
            opacity=0.7,
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=signals.index,
            y=signals["ma_long"],
            mode="lines",
            name="MA long",
            opacity=0.7,
        ),
        row=1,
        col=1,
    )

    buy_idx = signals.index[signals["bull_cross"]]
    sell_idx = signals.index[signals["bear_cross"]]

    fig.add_trace(
        go.Scatter(
            x=buy_idx,
            y=df.loc[buy_idx, "Low"],
            mode="markers",
            name="Buy",
            marker=dict(
                symbol="triangle-up",
                size=12,
            ),
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=sell_idx,
            y=df.loc[sell_idx, "High"],
            mode="markers",
            name="Sell",
            marker=dict(
                symbol="triangle-down",
                size=12,
            ),
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=cum_return.index,
            y=cum_return.values,
            mode="lines",
            name="Cumulative return",
        ),
        row=2,
        col=1,
    )

    fig.update_layout(
        title=title,
        xaxis_rangeslider_visible=False,
        legend_orientation="h",
    )

    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Return", tickformat=".0%", row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)

    fig.show()

In [16]:
plot_stocks_and_revenue(df)

In [24]:
from sko.GA import GA


def objective(x):
    short = int(round(x[0]))
    long = int(round(x[1]))

    if short >= long:
        return 1e9

    res = backtest_ma_crossover(
        df=df,
        short_window=short,
        long_window=long,
        long_only=True,
        fee_bps=10.0,
        initial_cash=10_000.0,
    )

    sharpe = res["metrics"]["sharpe"]
    if sharpe is None or not np.isfinite(sharpe):
        return 1e9

    return -sharpe


ga = GA(
    func=objective,
    n_dim=2,
    size_pop=80,
    max_iter=60,
    prob_mut=0.2,
    lb=[5, 20],
    ub=[50, 200],
    precision=[1, 1],
)

best_x, best_y = ga.run()
best_short, best_long = map(lambda v: int(round(v)), best_x)

best_short, best_long, best_y

(7, 20, array([-1.01109103]))

In [27]:
plot_stocks_and_revenue(
    df,
    short_window=best_short,
    long_window=best_long,
    title="MA Crossover + Returns after ACO",
)